In [0]:
%run ../connection

In [0]:
%run ./config

In [0]:
def run_autoloader(
    endpoint_name     : str,
    landing_subfolder : str,
    bronze_table_path : str,
) -> dict:
    """
    Run Databricks Autoloader for one CoinGecko endpoint.
    """
    checkpoint_path = f"{CHECKPOINT_ROOT}/{endpoint_name}"
    schema_path     = f"{SCHEMA_ROOT}/{endpoint_name}"
    landing_path    = f"{LANDING_ROOT}/{landing_subfolder}"
 
    logger.info(f"  ┌─ Autoloader: {endpoint_name}")
    logger.info(f"  │  Landing  : {landing_path}")
    logger.info(f"  │  Bronze   : {bronze_table_path}")
    logger.info(f"  │  Checkpoint: {checkpoint_path}")
 
    # ── READ STREAM ───────────────────────────────────────────────────────────
    raw_stream = (
        spark.readStream
             .format("cloudFiles")
             .options(**{
                 "cloudFiles.format"           : "json",
 
                 "cloudFiles.schemaLocation"   : schema_path,
 
                 "cloudFiles.inferColumnTypes" : "true",
 
                 "multiLine"                   : "true",
 
                 "recursiveFileLookup"         : "true",
 
                 "pathGlobFilter"              : "*.json",
             })
             .load(landing_path)
    )
 
    # ── ENRICH WITH BRONZE AUDIT COLUMNS 
    enriched_stream = (
        raw_stream
        .withColumn(
            "bronze_loaded_at",
            F.lit(NOW_UTC.isoformat()).cast(TimestampType())
        )
        .withColumn(
            "source_file_path",
            F.col("_metadata.file_path")   
        )
        .withColumn(
            "source_file_name",
            F.element_at(F.split(F.col("_metadata.file_path"), "/"), -1)
        )
    )
 
    # ── WRITE STREAM → BRONZE DELTA (path-based, NO catalog) ─────────────────
    query = (
        enriched_stream
        .writeStream
        .format("delta")
        .outputMode("append")                       
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")              
        .trigger(availableNow=True)                 
        .start(bronze_table_path)                 
    )

    query.awaitTermination()
 
    # ── READ BACK TO GET STATS (path-based) ──────────────────────────────────
    df    = spark.read.format("delta").load(bronze_table_path)
    count = df.count()
    cols  = df.columns
 
    logger.info(f"  └─ ✓ Done | rows: {count:,} | columns: {len(cols)}")
    return {"path": bronze_table_path, "row_count": count, "columns": cols}
 
 

In [0]:

# CELL 4 — OPTIMIZE HELPER
 
def optimize_delta(bronze_table_path: str, zorder_col: str, label: str) -> None:
    """
    Run OPTIMIZE + Z-ORDER on a Bronze Delta table accessed by ADLS path.
    Uses the delta.`path` syntax — no catalog or metastore required.
    """
    logger.info(f"  Optimizing {label} (Z-ORDER BY {zorder_col}) ...")
 
    spark.sql(f"OPTIMIZE delta.`{bronze_table_path}` ZORDER BY ({zorder_col})")
 
    logger.info(f"  ✓ {label} optimized")
 
 

In [0]:
# CELL 5 — RUN AUTOLOADER FOR ALL 5 ENDPOINTS
 
results = {}
 
# ── 1/5: coins/markets 
logger.info("=" * 70)
logger.info("AUTOLOADER 1/5: coins_markets")
results["coins_markets_raw"] = run_autoloader(
    endpoint_name     = "coins_markets",
    landing_subfolder = "coins_markets",
    bronze_table_path = f"{BRONZE_ROOT}/coins_markets_raw",
)
 
# ── 2/5: ohlc 
logger.info("=" * 70)
logger.info("AUTOLOADER 2/5: ohlc (all coins, recursive subfolders)")
results["ohlc_raw"] = run_autoloader(
    endpoint_name     = "ohlc",
    landing_subfolder = "ohlc",
    bronze_table_path = f"{BRONZE_ROOT}/ohlc_raw",
)
 
# ── 3/5: market_chart 
logger.info("=" * 70)
logger.info("AUTOLOADER 3/5: market_chart (all coins, recursive subfolders)")
results["market_chart_raw"] = run_autoloader(
    endpoint_name     = "market_chart",
    landing_subfolder = "market_chart",
    bronze_table_path = f"{BRONZE_ROOT}/market_chart_raw",
)
 
# ── 4/5: trending 
logger.info("=" * 70)
logger.info("AUTOLOADER 4/5: trending")
results["trending_raw"] = run_autoloader(
    endpoint_name     = "trending",
    landing_subfolder = "trending",
    bronze_table_path = f"{BRONZE_ROOT}/trending_raw",
)
 
# ── 5/5: global 
logger.info("=" * 70)
logger.info("AUTOLOADER 5/5: global")
results["global_raw"] = run_autoloader(
    endpoint_name     = "global",
    landing_subfolder = "global",
    bronze_table_path = f"{BRONZE_ROOT}/global_raw",
)
 
 

In [0]:

# CELL 6 — VALIDATION
logger.info("=" * 70)
logger.info("VALIDATION: Checking all 5 Bronze Delta tables")
 
REQUIRED_ENVELOPE_COLS = {"pipeline_run_id", "ingestion_timestamp", "source_endpoint"}
REQUIRED_BRONZE_COLS   = {"bronze_loaded_at", "source_file_path", "source_file_name"}
 
validation_summary = {}
failed_tables      = []
 
for table_name, result in results.items():
    col_set          = set(result["columns"])
    row_count        = result["row_count"]
    missing_envelope = REQUIRED_ENVELOPE_COLS - col_set
    missing_bronze   = REQUIRED_BRONZE_COLS   - col_set
    has_data         = row_count > 0
    passed           = has_data and not missing_envelope and not missing_bronze
    status           = "PASS" if passed else "FAIL"
 
    validation_summary[table_name] = {
        "status"      : status,
        "row_count"   : row_count,
        "missing_cols": list(missing_envelope | missing_bronze),
    }
 
    logger.info(
        f"  [{status}] {table_name:<25} "
        f"rows: {row_count:>6,}  "
        f"missing: {list(missing_envelope | missing_bronze) or 'none'}"
    )
 
    if not passed:
        failed_tables.append(table_name)
 
if failed_tables:
    raise ValueError(
        f"Bronze validation FAILED for: {failed_tables}\n"
        f"Details:\n{json.dumps(validation_summary, indent=2)}"
    )
 
logger.info("  ✓ All 5 Bronze tables passed validation")
 
 

In [0]:
%sql
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;

In [0]:
# CELL 7 — OPTIMIZE + Z-ORDER
 
logger.info("=" * 70)
logger.info("OPTIMIZE: Compacting Bronze Delta tables")
 
optimize_delta(f"{BRONZE_ROOT}/coins_markets_raw", "pipeline_run_id", "coins_markets_raw")
optimize_delta(f"{BRONZE_ROOT}/ohlc_raw",           "coin_id",         "ohlc_raw")
optimize_delta(f"{BRONZE_ROOT}/market_chart_raw",   "coin_id",         "market_chart_raw")
optimize_delta(f"{BRONZE_ROOT}/trending_raw",       "pipeline_run_id", "trending_raw")
optimize_delta(f"{BRONZE_ROOT}/global_raw",         "pipeline_run_id", "global_raw")
 
logger.info("  ✓ All tables optimized")
 

In [0]:

# CELL 8 — COMPLETION SUMMARY

summary = {
    "event"            : "bronze_autoloader_complete",
    "run_timestamp_utc": NOW_UTC.isoformat(),
    "tables_loaded"    : len(results),
    "row_counts"       : {t: r["row_count"] for t, r in results.items()},
    "validation"       : {t: v["status"]    for t, v in validation_summary.items()},
    "status"           : "SUCCESS",
}
 
logger.info("=" * 70)
logger.info("BRONZE AUTOLOADER COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<28}: {v}")
logger.info("=" * 70)
logger.info("")
logger.info("HOW TO READ THESE TABLES IN SILVER NOTEBOOK:")
logger.info("  # No catalog or database needed — just the ADLS path")
logger.info("  df = spark.read.format('delta').load(")
logger.info(f"      'abfss://bronze@{ADLS_NAME}.dfs.core.windows.net/coins_markets_raw'")
logger.info("  )")
 